# Task 2 - Generate Descriptions

## System Prompt

The system prompt below instructs the model to produce a persuasive 50–90 word product description based on the provided information (name, structured attributes, material, warranty).

Key design decisions:
- **Explicit length constraint** with instruction to count carefully
- **Grounding rule** to prevent hallucination of facts not in the input
- **Tone, fluency, grammar** rules aligned with the Task 1 rubric criteria
- **One few-shot example** to demonstrate the expected output format and style
- Output-only instruction to prevent the model from adding commentary

In [ ]:
SYSTEM_PROMPT = (
    "You are a professional e-commerce copywriter. Your task is to write a persuasive product description for a given product.\n\n"
    "Rules you MUST follow:\n\n"
    "1. LENGTH: Write exactly 50 to 90 words. Count carefully. Do not go below 50 or above 90.\n\n"
    "2. GROUNDING: Use ONLY the information provided (product name, attributes, material, warranty). "
    "Do NOT invent specifications, certifications, dimensions, brand claims, or any facts not given. "
    "You may use generic marketing phrases like \"perfect for everyday use\" but never fabricate concrete facts.\n\n"
    "3. TONE: Write in a friendly, credible sales voice. Focus on customer benefits and value. "
    "Do not use clickbait, exaggerated promises, or robotic language.\n\n"
    "4. FLUENCY: Write clear, natural sentences that flow logically. No awkward phrasing or abrupt jumps.\n\n"
    "5. GRAMMAR: Use correct spelling, punctuation, and consistent tense.\n\n"
    "Output ONLY the product description text. No headings, no labels, no word count, no extra commentary.\n\n"
    "Example:\n"
    "Input:\n"
    "- Name: Yeti Rambler 20 oz Tumbler\n"
    "- Attributes: features: double-wall vacuum insulated, MagSlider lid; dimensions: compact\n"
    "- Material: kitchen-grade stainless steel\n"
    "- Warranty: 5-year warranty\n\n"
    "Output:\n"
    "Keep your drinks at the perfect temperature all day with the Yeti Rambler 20 oz Tumbler. "
    "Crafted from kitchen-grade stainless steel with double-wall vacuum insulation, this compact tumbler delivers outstanding heat and cold retention. "
    "The convenient MagSlider lid prevents spills while keeping your beverage easy to sip. "
    "Backed by a generous 5-year warranty, this reliable tumbler is built to last through every adventure and daily commute."
)

## Model

Using `google/gemma-2-9b-it-fast` from Nebius Token Factory (fast flavor of Gemma-2-9b-it).

In [ ]:
import os
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY")
BASE_URL       = "https://api.studio.nebius.com/v1/"
MODEL          = "google/gemma-2-9b-it-fast"

client = OpenAI(base_url=BASE_URL, api_key=NEBIUS_API_KEY)

## Generation Function

In [ ]:
def generate_description(user_prompt: str) -> dict:
    """
    Calls the model and returns:
    {
        'generated_description': str,
        'latency_ms': float,
        'input_tokens': int,
        'output_tokens': int
    }
    """
    start = time.time()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=150,
    )

    latency_ms = (time.time() - start) * 1000

    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms":            round(latency_ms, 2),
        "input_tokens":          response.usage.prompt_tokens,
        "output_tokens":         response.usage.completion_tokens,
    }

## Run Through Dataset

In [ ]:
df = pd.read_csv("Assignment_01_product_dataset.csv")

records = []
for _, row in df.iterrows():
    user_prompt = (
        f"Name: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}"
    )
    record = generate_description(user_prompt)
    records.append(record)

metrics_df = pd.DataFrame(records)
out_df     = pd.concat([df, metrics_df], axis=1)

## Cost Calculation and Save

Nebius pricing for `gemma-2-9b-it` (base flavor):  
- Input tokens: **$0.03 per 1M tokens**  
- Output tokens: **$0.09 per 1M tokens**

In [ ]:
INPUT_PRICE  = 0.03 / 1_000_000
OUTPUT_PRICE = 0.09 / 1_000_000

out_df["cost_usd"]   = out_df["input_tokens"] * INPUT_PRICE + out_df["output_tokens"] * OUTPUT_PRICE
out_df["word_count"] = out_df["generated_description"].apply(lambda x: len(x.split()))

# Blank rubric columns for Task 3 manual evaluation
for col in ["fluency", "grammar", "tone", "length", "grounding", "latency_rating", "cost_rating"]:
    out_df[col] = ""
out_df["final_score"] = ""

out_df.to_excel("assignment_01.xlsx", index=False, engine="openpyxl")
print(f"Saved {len(out_df)} rows to assignment_01.xlsx")
print(f"Mean latency: {out_df['latency_ms'].mean():.0f}ms")
print(f"Mean cost per call: ${out_df['cost_usd'].mean():.6f}")
print(f"Total cost: ${out_df['cost_usd'].sum():.6f}")